# Challenge 1: House Prices Prediction (Ames Housing) — phiên bản làm lại
**Nhóm 4 – Lớp DCT124C1 – Trường Đại học Sài Gòn** | GVHD: TS. Đỗ Như Tài  
Thành viên: Phạm Minh Khôi (3124411135), Trương Thuận Phú (3124411234), Nguyễn Mạnh Thắng (3124411283), Nguyễn Hoàng Quân (3124411250)

Notebook này chạy **một lượt từ đầu đến cuối** và sinh ra toàn bộ số liệu, hình (`figs/`), bảng (`outputs/`) và `submission.csv` dùng trong paper, slide, poster.

Đặc điểm của quy trình:
1. Chuẩn hóa, chọn đặc trưng (SelectKBest, RF importance) được đặt **bên trong `Pipeline`** nên mỗi fold chỉ học từ tập train của fold đó (không rò rỉ dữ liệu).
2. Tiền xử lý khớp đúng mô tả trong paper (LotFrontage theo Neighborhood, các cột diện tích thiếu → 0).
3. Có **ablation** cho log-transform và loại outlier để kiểm chứng định lượng thay vì chỉ khẳng định.
4. Thêm bước Feature Engineering + xử lý độ lệch + tinh chỉnh siêu tham số + blending, có CV riêng cho mô hình nộp cuối.
5. Mọi hình đều sinh từ code ở đây; hình Actual vs Predicted dùng dự đoán **out-of-fold** (không phải dự đoán trên tập train).

In [1]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
SEED = 42
for d in ('figs', 'outputs'): os.makedirs(d, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
NAVY, BLUE, ORANGE, GREEN, GREY = '#1F3A5F', '#4C78A8', '#F58518', '#54A24B', '#8C8C8C'
RES = {}   # gom số liệu cho paper/slide/poster
print('Môi trường sẵn sàng.')

Môi trường sẵn sàng.


## 1. Dữ liệu

In [2]:
train = pd.read_csv('data/train.csv'); test = pd.read_csv('data/test.csv')
print('Train:', train.shape, '| Test:', test.shape)
feat = train.drop(columns=['Id', 'SalePrice'])
n_num = feat.select_dtypes('number').shape[1]; n_cat = feat.shape[1] - n_num
print(f'Số biến dự báo: {feat.shape[1]} (số học {n_num}, định tính {n_cat})')
miss = (feat.isna().mean() * 100).sort_values(ascending=False)
print('Top biến khuyết nhiều nhất (%):'); print(miss.head(6).round(1))
RES.update(n_train=len(train), n_test=len(test), n_feat=feat.shape[1], n_num=n_num, n_cat=n_cat,
           miss_top={k: round(v, 1) for k, v in miss.head(6).items()})

Train: (1460, 81) | Test: (1459, 80)
Số biến dự báo: 79 (số học 36, định tính 43)
Top biến khuyết nhiều nhất (%):
PoolQC         99.5
MiscFeature    96.3
Alley          93.8
Fence          80.8
MasVnrType     59.7
FireplaceQu    47.3
dtype: float64


## 2. Biến mục tiêu và điểm ngoại lai

In [3]:
sk0 = train.SalePrice.skew(); sk1 = np.log1p(train.SalePrice).skew()
print(f'Skewness SalePrice: {sk0:.2f}  →  log1p: {sk1:.2f}')
out_idx = train[(train.GrLivArea > 4000) & (train.SalePrice < 300000)].index
print(train.loc[out_idx, ['Id', 'GrLivArea', 'SalePrice', 'SaleCondition']])
RES.update(skew_raw=round(sk0, 2), skew_log=round(sk1, 2), outlier_ids=train.loc[out_idx, 'Id'].tolist(),
           outlier_cond=train.loc[out_idx, 'SaleCondition'].tolist())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(train.SalePrice, bins=50, color=BLUE, edgecolor='white'); ax[0].set_title(f'SalePrice gốc (skewness = {sk0:.2f})'); ax[0].set_xlabel('SalePrice ($)')
ax[1].hist(np.log1p(train.SalePrice), bins=50, color=GREEN, edgecolor='white'); ax[1].set_title(f'log1p(SalePrice) (skewness = {sk1:.2f})'); ax[1].set_xlabel('log(1 + SalePrice)')
for a in ax: a.set_ylabel('Số căn nhà')
plt.tight_layout(); plt.savefig('figs/fig2_target_distribution.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.scatter(train.GrLivArea, train.SalePrice, s=10, alpha=.45, color=BLUE, label='Các căn nhà còn lại')
o = train.loc[out_idx]; ax.scatter(o.GrLivArea, o.SalePrice, s=60, color='crimson', label='2 điểm ngoại lai (Id %s)' % ', '.join(map(str, o.Id)))
ax.axvline(4000, ls='--', color=GREY); ax.axhline(300000, ls=':', color=GREY)
ax.set_xlabel('GrLivArea (sq ft)'); ax.set_ylabel('SalePrice ($)'); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig('figs/fig3_outliers.png', dpi=200); plt.show()

Skewness SalePrice: 1.88  →  log1p: 0.12
        Id  GrLivArea  SalePrice SaleCondition
523    524       4676     184750       Partial
1298  1299       5642     160000       Partial


## 3. Tiền xử lý
- Loại 2 outlier (GrLivArea > 4000 và SalePrice < 300.000$; cả hai đều là giao dịch `Partial`).
- Biến phân loại: `NA` → `'None'`, mã hóa nhãn (LabelEncoder tương đương; tập nhãn lấy từ train + test chỉ để đồng bộ mã, không dùng target).
- Biến số: cột diện tích/đếm của tiện ích không tồn tại → 0; `LotFrontage` → median theo Neighborhood (tính trên train); các cột còn lại → median của train.
- Chuẩn hóa `StandardScaler` được đặt trong Pipeline (fit riêng cho từng fold).

In [4]:
ZERO = ['MasVnrArea','GarageYrBlt','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','GarageCars','GarageArea']

def load(drop_outliers=True):
    tr = pd.read_csv('data/train.csv'); te = pd.read_csv('data/test.csv')
    if drop_outliers:
        tr = tr.drop(tr[(tr.GrLivArea > 4000) & (tr.SalePrice < 300000)].index).reset_index(drop=True)
    y = tr.SalePrice.values; ids = te.Id.values
    X = tr.drop(['Id', 'SalePrice'], axis=1); T = te.drop(['Id'], axis=1)
    cat = X.select_dtypes(exclude='number').columns.tolist(); num = X.select_dtypes('number').columns.tolist()
    med = X.groupby('Neighborhood').LotFrontage.median(); g = X.LotFrontage.median()
    for D in (X, T): D['LotFrontage'] = D.LotFrontage.fillna(D.Neighborhood.map(med)).fillna(g)
    for c in num:
        for D in (X, T): D[c] = D[c].fillna(0 if c in ZERO else X[c].median())
    for c in cat:
        X[c] = X[c].fillna('None').astype(str); T[c] = T[c].fillna('None').astype(str)
        mp = {k: i for i, k in enumerate(sorted(set(X[c]) | set(T[c])))}
        X[c] = X[c].map(mp); T[c] = T[c].map(mp)
    return X, y, T, ids

X, y, T, ids = load(True); ylog = np.log1p(y)
print('X:', X.shape, '| T:', T.shape, '| thiếu sau xử lý:', int(X.isna().sum().sum()), int(T.isna().sum().sum()))
RES['n_train_clean'] = len(X)

X: (1458, 79) | T: (1459, 79) | thiếu sau xử lý: 0 0


## 4. Tương quan

In [5]:
raw = pd.read_csv('data/train.csv').drop(index=out_idx).reset_index(drop=True)
num_raw = raw.drop(columns=['Id']).select_dtypes('number')      # chỉ biến số thật (không dùng nhãn mã hóa), tương quan theo cặp
corr = num_raw.corr()
top = corr['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False).head(11).index.tolist()
r_top = {c: round(corr.loc[c, 'SalePrice'], 2) for c in top}
print('Tương quan Pearson với SalePrice:', r_top)
pairs = [('GarageCars','GarageArea'),('TotalBsmtSF','1stFlrSF'),('YearBuilt','GarageYrBlt')]
r_pairs = {f'{a}~{b}': round(corr.loc[a, b], 2) for a, b in pairs}; print('Cặp đa cộng tuyến:', r_pairs)
RES.update(r_top=r_top, r_pairs=r_pairs)
cols = ['SalePrice'] + top[:10]
fig, ax = plt.subplots(figsize=(7.5, 6.2))
sns.heatmap(corr.loc[cols, cols], annot=True, fmt='.2f', cmap='Blues', annot_kws={'size': 8}, cbar_kws={'shrink': .8}, ax=ax)
plt.tight_layout(); plt.savefig('figs/fig4_correlation.png', dpi=200); plt.show()

Tương quan Pearson với SalePrice: {'OverallQual': np.float64(0.8), 'GrLivArea': np.float64(0.73), 'TotalBsmtSF': np.float64(0.65), 'GarageCars': np.float64(0.64), '1stFlrSF': np.float64(0.63), 'GarageArea': np.float64(0.63), 'FullBath': np.float64(0.56), 'TotRmsAbvGrd': np.float64(0.54), 'YearBuilt': np.float64(0.52), 'YearRemodAdd': np.float64(0.51), 'GarageYrBlt': np.float64(0.49)}
Cặp đa cộng tuyến: {'GarageCars~GarageArea': np.float64(0.89), 'TotalBsmtSF~1stFlrSF': np.float64(0.8), 'YearBuilt~GarageYrBlt': np.float64(0.83)}


## 5. Thực nghiệm chính: 3 tập đặc trưng × 5 mô hình (5-fold CV, không rò rỉ)

In [6]:
kf = KFold(5, shuffle=True, random_state=SEED)
MODELS = {
    'Ridge': lambda: Ridge(alpha=10.0),
    'SVR': lambda: SVR(C=1.0, epsilon=0.1),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Gradient Boosting': lambda: GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, random_state=SEED),
    'XGBoost': lambda: XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=SEED, n_jobs=-1, eval_metric='rmse'),
}
def selector(name):
    if name == 'All (79)': return 'passthrough'
    if name == 'SelectKBest (30)': return SelectKBest(f_regression, k=30)
    return SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1), max_features=25, threshold=-np.inf)

rows = []
for fs in ['All (79)', 'SelectKBest (30)', 'RF Importance (25)']:
    for mn, mk in MODELS.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('select', selector(fs)), ('model', mk())])
        t0 = time.time(); r = cross_validate(pipe, X, ylog, cv=kf, scoring=('neg_mean_squared_error', 'r2'))
        rm = np.sqrt(-r['test_neg_mean_squared_error'])
        rows.append(dict(FeatureSet=fs, Model=mn, RMSE=rm.mean(), RMSE_Std=rm.std(), R2=r['test_r2'].mean(), CV_time_s=time.time() - t0))
bench = pd.DataFrame(rows); bench.to_csv('outputs/benchmark_results.csv', index=False)
print(bench.round(4).to_string(index=False))
RES['bench'] = bench.round(4).to_dict('records')
best = bench.loc[bench.RMSE.idxmin()]; print('\nTốt nhất:', best.FeatureSet, best.Model, round(best.RMSE, 4))

pv = bench.pivot(index='Model', columns='FeatureSet', values='RMSE')[['All (79)', 'SelectKBest (30)', 'RF Importance (25)']].loc[list(MODELS)]
fig, ax = plt.subplots(figsize=(9, 4.4)); w = .26
for i, (c, col) in enumerate(zip(pv.columns, [NAVY, BLUE, ORANGE])):
    b = ax.bar(np.arange(len(pv)) + (i - 1) * w, pv[c], w, label=c, color=col)
    for rect in b: ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + .002, f'{rect.get_height():.3f}', ha='center', fontsize=7)
ax.set_xticks(np.arange(len(pv))); ax.set_xticklabels(pv.index); ax.set_ylabel('RMSE trên log1p(SalePrice) (thấp hơn là tốt hơn)'); ax.set_ylim(.10, .185); ax.legend(frameon=False, ncol=3, loc='upper left')
plt.tight_layout(); plt.savefig('figs/fig6_rmse_comparison.png', dpi=200); plt.show()

        FeatureSet             Model   RMSE  RMSE_Std     R2  CV_time_s
          All (79)             Ridge 0.1218    0.0089 0.9065     0.1940
          All (79)               SVR 0.1703    0.0190 0.8177     0.5046
          All (79)     Random Forest 0.1390    0.0085 0.8777     2.0934
          All (79) Gradient Boosting 0.1269    0.0098 0.8981     4.1580
          All (79)           XGBoost 0.1255    0.0095 0.9004     0.7300
  SelectKBest (30)             Ridge 0.1354    0.0067 0.8843     0.1524
  SelectKBest (30)               SVR 0.1559    0.0113 0.8475     0.5264
  SelectKBest (30)     Random Forest 0.1461    0.0098 0.8645     1.5905
  SelectKBest (30) Gradient Boosting 0.1404    0.0070 0.8753     2.6798
  SelectKBest (30)           XGBoost 0.1401    0.0065 0.8757     0.5556
RF Importance (25)             Ridge 0.1287    0.0088 0.8953     2.5157
RF Importance (25)               SVR 0.1467    0.0164 0.8649     3.0713
RF Importance (25)     Random Forest 0.1385    0.0090 0.8786    

### Độ quan trọng đặc trưng (Random Forest, MSE-based impurity)

In [7]:
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1).fit(StandardScaler().fit_transform(X), ylog)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10).round(4)); print('Top-25 chiếm', round(imp.head(25).sum() * 100, 1), '% tổng độ quan trọng')
RES['imp_top10'] = imp.head(10).round(4).to_dict(); RES['imp_top25_share'] = round(imp.head(25).sum() * 100, 1)
RES['top25'] = imp.head(25).index.tolist()
h = imp.head(15)[::-1]
fig, ax = plt.subplots(figsize=(6.5, 4.8)); ax.barh(h.index, h.values, color=NAVY)
ax.set_xlabel('Độ quan trọng (giảm MSE trung bình)'); plt.tight_layout(); plt.savefig('figs/fig5_importance.png', dpi=200); plt.show()

OverallQual    0.5496
GrLivArea      0.1098
TotalBsmtSF    0.0455
GarageCars     0.0402
GarageArea     0.0284
1stFlrSF       0.0220
BsmtFinSF1     0.0209
GarageType     0.0152
YearBuilt      0.0149
OverallCond    0.0129
dtype: float64
Top-25 chiếm 94.8 % tổng độ quan trọng


## 6. Ablation: log-transform và loại outlier có thật sự cần thiết?

In [8]:
def rmse_log(p, yy): return np.sqrt(np.mean((np.log1p(np.clip(p, 1, None)) - np.log1p(yy)) ** 2))
ab = []
for drop in (True, False):
    Xa, ya, _, _ = load(drop); keep = ~((Xa.GrLivArea > 4000) & (ya < 300000))
    for tgt in ('log1p', 'raw'):
        for mn, mk in {'Ridge': lambda: make_pipeline(StandardScaler(), Ridge(alpha=10)),
                       'GBDT': lambda: GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, random_state=SEED)}.items():
            yt = np.log1p(ya) if tgt == 'log1p' else ya
            p = cross_val_predict(mk(), Xa, yt, cv=kf); p = np.expm1(p) if tgt == 'log1p' else p
            ab.append(dict(Outlier='đã loại' if drop else 'giữ lại', Target=tgt, Model=mn, RMSE_log=rmse_log(p[keep.values], ya[keep.values])))
abl = pd.DataFrame(ab); abl.to_csv('outputs/ablation_results.csv', index=False)
print('(Điểm số luôn tính trên cùng 1.458 căn nhà để so sánh công bằng)'); print(abl.round(4).to_string(index=False))
RES['ablation'] = abl.round(4).to_dict('records')

(Điểm số luôn tính trên cùng 1.458 căn nhà để so sánh công bằng)
Outlier Target Model  RMSE_log
đã loại  log1p Ridge    0.1221
đã loại  log1p  GBDT    0.1272
đã loại    raw Ridge    0.1574
đã loại    raw  GBDT    0.1298
giữ lại  log1p Ridge    0.1388
giữ lại  log1p  GBDT    0.1272
giữ lại    raw Ridge    0.3520
giữ lại    raw  GBDT    0.1312


## 7. Feature Engineering, xử lý độ lệch, tinh chỉnh và blending
Từ kết quả mục 5, Ridge trên đầy đủ đặc trưng đã rất mạnh, nên nhóm cải thiện dữ liệu đầu vào (đặc trưng mới, log các biến lệch) và kết hợp mô hình tuyến tính có điều chuẩn với boosting.

In [9]:
def add_fe(D):
    D = D.copy()
    D['TotalSF'] = D.TotalBsmtSF + D['1stFlrSF'] + D['2ndFlrSF']
    D['TotalBath'] = D.FullBath + .5 * D.HalfBath + D.BsmtFullBath + .5 * D.BsmtHalfBath
    D['HouseAge'] = D.YrSold - D.YearBuilt; D['RemodAge'] = D.YrSold - D.YearRemodAdd
    D['TotalPorchSF'] = D.OpenPorchSF + D.EnclosedPorch + D['3SsnPorch'] + D.ScreenPorch + D.WoodDeckSF
    D['HasPool'] = (D.PoolArea > 0).astype(int); D['Has2ndFloor'] = (D['2ndFlrSF'] > 0).astype(int)
    D['QualxArea'] = D.OverallQual * D.GrLivArea
    return D
Xf, Tf = add_fe(X), add_fe(T)
sk = Xf.apply(lambda s: abs(skew(s)))
skew_cols = [c for c in Xf.columns if sk[c] > .75 and Xf[c].min() >= 0 and Xf[c].nunique() > 10]
for D in (Xf, Tf): D[skew_cols] = np.log1p(D[skew_cols])
print('Số đặc trưng:', Xf.shape[1], '| biến được log1p do lệch:', len(skew_cols))
RES.update(n_feat_fe=Xf.shape[1], n_skew_cols=len(skew_cols))

def cv_scores(model, D):
    r = cross_validate(model, D, ylog, cv=kf, scoring=('neg_mean_squared_error', 'r2')); rm = np.sqrt(-r['test_neg_mean_squared_error'])
    return rm.mean(), rm.std(), r['test_r2'].mean()
step = []
step.append(('Ridge (α=10), 79 biến gốc', *cv_scores(make_pipeline(StandardScaler(), Ridge(alpha=10)), X)))
gr = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': [1, 3, 10, 30, 100, 300]}, cv=kf, scoring='neg_mean_squared_error').fit(Xf, ylog)
gl = GridSearchCV(make_pipeline(StandardScaler(), Lasso(max_iter=20000)), {'lasso__alpha': [3e-4, 5e-4, 1e-3, 2e-3, 4e-3, 8e-3]}, cv=kf, scoring='neg_mean_squared_error').fit(Xf, ylog)
print('Ridge α tốt nhất:', gr.best_params_, '| Lasso α tốt nhất:', gl.best_params_)
ridge, lasso = gr.best_estimator_, gl.best_estimator_
gbdt = GradientBoostingRegressor(n_estimators=600, learning_rate=.03, max_depth=3, subsample=.7, max_features=.5, random_state=SEED)
step.append(('Ridge tinh chỉnh + FE + log biến lệch', *cv_scores(ridge, Xf)))
step.append(('Lasso tinh chỉnh + FE + log biến lệch', *cv_scores(lasso, Xf)))
step.append(('GBDT tinh chỉnh + FE + log biến lệch', *cv_scores(gbdt, Xf)))

oof = {n: cross_val_predict(m, Xf, ylog, cv=kf) for n, m in (('Ridge', ridge), ('Lasso', lasso), ('GBDT', gbdt))}
W = {'Ridge': .25, 'Lasso': .25, 'GBDT': .5}
blend = sum(W[n] * oof[n] for n in W)
fold_rm = [np.sqrt(np.mean((blend[v] - ylog[v]) ** 2)) for _, v in kf.split(Xf)]
step.append(('Blend: 0.25 Ridge + 0.25 Lasso + 0.5 GBDT', np.mean(fold_rm), np.std(fold_rm), r2_score(ylog, blend)))
stage = pd.DataFrame(step, columns=['Cấu hình', 'RMSE', 'RMSE_Std', 'R2']); stage.to_csv('outputs/final_stage_results.csv', index=False)
print(stage.round(4).to_string(index=False))
RES['stage'] = stage.round(4).to_dict('records'); RES.update(ridge_alpha=gr.best_params_['ridge__alpha'], lasso_alpha=gl.best_params_['lasso__alpha'], blend_w=W)
RES['lasso_nonzero'] = int((lasso[-1].coef_ != 0).sum())

fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.scatter(ylog, blend, s=9, alpha=.45, color=BLUE)
lo, hi = ylog.min() - .05, ylog.max() + .05; ax.plot([lo, hi], [lo, hi], '--', color='crimson', label='y = ŷ')
ax.set_xlabel('log(1 + giá thực tế)'); ax.set_ylabel('log(1 + giá dự đoán, out-of-fold)')
ax.set_title(f'Dự đoán out-of-fold của mô hình blend (R² = {r2_score(ylog, blend):.3f})', fontsize=10); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig('figs/fig7_actual_vs_pred.png', dpi=200); plt.show()
res_df = ylog - blend; RES['oof_resid_std'] = round(float(res_df.std()), 4)

Số đặc trưng: 87 | biến được log1p do lệch: 21


Ridge α tốt nhất: {'ridge__alpha': 100} | Lasso α tốt nhất: {'lasso__alpha': 0.002}


                                 Cấu hình   RMSE  RMSE_Std     R2
                Ridge (α=10), 79 biến gốc 0.1218    0.0089 0.9065
    Ridge tinh chỉnh + FE + log biến lệch 0.1197    0.0067 0.9095
    Lasso tinh chỉnh + FE + log biến lệch 0.1188    0.0081 0.9106
     GBDT tinh chỉnh + FE + log biến lệch 0.1161    0.0068 0.9152
Blend: 0.25 Ridge + 0.25 Lasso + 0.5 GBDT 0.1127    0.0076 0.9202


## 8. Huấn luyện mô hình cuối và xuất `submission.csv`

In [10]:
final = {n: m.fit(Xf, ylog) for n, m in (('Ridge', ridge), ('Lasso', lasso), ('GBDT', gbdt))}
pred_log = sum(W[n] * final[n].predict(Tf) for n in W)
n_clip = int(((pred_log > ylog.max()) | (pred_log < ylog.min())).sum())   # mô hình tuyến tính ngoại suy: chặn trong khoảng giá đã thấy ở train
pred = np.expm1(np.clip(pred_log, ylog.min(), ylog.max())); RES['n_clipped'] = n_clip; print('Số dự đoán bị chặn trong khoảng giá train:', n_clip)
sub = pd.DataFrame({'Id': ids, 'SalePrice': pred}); sub.to_csv('outputs/submission.csv', index=False)
ss = pd.read_csv('data/sample_submission.csv')
assert len(sub) == len(ss) == 1459 and (sub.Id.values == ss.Id.values).all() and sub.SalePrice.notna().all() and (sub.SalePrice > 0).all()
print('Đã xuất outputs/submission.csv:', sub.shape); print(sub.SalePrice.describe().round(0))
old = pd.read_csv('data/submission_old.csv') if os.path.exists('data/submission_old.csv') else None
if old is not None:
    c = np.corrcoef(np.log(old.SalePrice), np.log(sub.SalePrice))[0, 1]; print('Tương quan (log) với submission cũ:', round(c, 4)); RES['corr_old_sub'] = round(float(c), 4)
RES['sub_stats'] = {k: round(float(v)) for k, v in sub.SalePrice.describe()[['min', '50%', 'mean', 'max']].items()}

Số dự đoán bị chặn trong khoảng giá train: 1
Đã xuất outputs/submission.csv: (1459, 2)
count      1459.0
mean     178165.0
std       76946.0
min       44699.0
25%      127352.0
50%      157565.0
75%      209556.0
max      755000.0
Name: SalePrice, dtype: float64


## 9. Sơ đồ quy trình (Hình 1)

In [11]:
fig, ax = plt.subplots(figsize=(11, 2.3)); ax.axis('off'); ax.set_xlim(0, 100); ax.set_ylim(3, 27)
boxes = [('1. Dữ liệu thô', '1.460 nhà · 79 biến'), ('2. Tiền xử lý', 'Loại 2 outlier · điền khuyết\nlog1p(SalePrice) · mã hóa'),
         ('3. Chọn đặc trưng', 'Tất cả 79 · SelectKBest 30\nRF importance 25'), ('4. Mô hình hóa', 'Ridge · SVR · RF\nGBDT · XGBoost'),
         ('5. Đánh giá', '5-fold CV (Pipeline)\nRMSE · R²'), ('6. Mô hình nộp', 'FE + blend Ridge/Lasso/GBDT\n→ submission.csv')]
w, gap = 13.6, 2.7
for i, (t, s) in enumerate(boxes):
    x = i * (w + gap) + 1
    ax.add_patch(plt.Rectangle((x, 6), w, 18, fc='#E8F0FB', ec=NAVY, lw=1.6))
    ax.text(x + w / 2, 19.5, t, ha='center', va='center', fontsize=9.5, fontweight='bold', color=NAVY)
    ax.text(x + w / 2, 12.5, s, ha='center', va='center', fontsize=7.2)
    if i < len(boxes) - 1: ax.annotate('', xy=(x + w + gap - .3, 15), xytext=(x + w + .3, 15), arrowprops=dict(arrowstyle='->', color=NAVY, lw=1.6))
plt.tight_layout(); plt.savefig('figs/fig1_pipeline.png', dpi=200, bbox_inches='tight', pad_inches=0.05); plt.show()

In [12]:
json.dump(RES, open('outputs/results_summary.json', 'w'), ensure_ascii=False, indent=1)
print('Đã lưu outputs/results_summary.json và', len(os.listdir('figs')), 'hình trong figs/')

Đã lưu outputs/results_summary.json và 7 hình trong figs/
